In [1]:
import geopandas as gpd
import pandas as pd

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
df = gpd.read_file('ii. health-facilities/Academics/senegal/senegal_full_facilitylist.csv')
start = len(df)

# Drop facilities with missing coordinates
df.dropna(subset=['latitude', 'longitude'],
        inplace = True)

df = df.loc[df.latitude != ""].copy()
print(f"{start - len(df)} facilities with missing geometries dropped.\n {len(df)} facilities remaining.")

# Drop original geometry column (is None)
if 'geometry' in df.columns:
        df.drop(columns = ['geometry'], inplace = True)

# Convert to gdf
gdf = gpd.GeoDataFrame(df, geometry = gpd.points_from_xy(df.longitude, df.latitude), crs='epsg:4326')

# Export
gdf.to_file("../reach/health-centres/Academics/senegal/hf.geojson")

7647 facilities with missing geometries dropped.
 5318 facilities remaining.


/Users/kt/anaconda3/envs/bigd/lib/python3.11/site-packages/geopandas/geodataframe.py:62: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  data.crs = crs
/Users/kt/anaconda3/envs/bigd/lib/python3.11/site-packages/geopandas/geodataframe.py:469: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  level.crs = crs


Create a version that conservatively assumes which endpoints offer family planning:
* pharmacies
* ASBEF locations
* Hospitals
* Private endpoints

In [3]:
fp_services_name = [
    'pharmacie',
    'asbef'
]

fp_services_type = [
    'hopital'
]

fp_fac_own = [
    'prive'
]

gdf = gpd.read_file('../reach/health-centres/Academics/senegal/hf.geojson')
start = len(gdf)

# only retain facilities with key names or facility types
gdf = gdf.loc[
            gdf.fac_name_orig.str.lower().str.contains("|".join(fp_services_name)) | 
            gdf.fac_type_orig.str.lower().str.contains("|".join(fp_services_type)) | 
            gdf.group_fac_own.str.lower().str.contains("|".join(fp_fac_own))
            ].copy()
print(f"{start - len(gdf)} facilities that did not meet inclusion criteria dropped.\n {len(gdf)} facilities remaining.")

# make CSV for academics folder
gdf['latitude'] = gdf.geometry.y
gdf['longitude'] = gdf.geometry.x
df = pd.DataFrame(gdf.drop(columns = 'geometry'))

# Export
df.to_csv("../reach/health-centres/Academics/senegal/hf_fp_services_guess.csv")
gdf.to_file("../reach/health-centres/Academics/senegal/hf_fp_services_guess.geojson")

4903 facilities that did not meet inclusion criteria dropped.
 415 facilities remaining.


/Users/kt/anaconda3/envs/bigd/lib/python3.11/site-packages/geopandas/array.py:344: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  self.crs = crs
/Users/kt/anaconda3/envs/bigd/lib/python3.11/site-packages/geopandas/geodataframe.py:64: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  data.array.crs = crs
/Users/kt/anaconda3/envs/bigd/lib/python3.11/site-packages/geopandas/geodataframe.py:467: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may 

In [4]:
len(gdf)

415